## DFU Risk Classification

This notebook implements a production-ready training pipeline for Diabetic Foot Ulcer (DFU) detection using three state-of-the-art architectures:
- **EfficientNetB0**: Efficient mobile-friendly architecture
- **ResNet50**: Deep residual learning baseline
- **ConvNeXt-Tiny**: Modern convolutional neural network

### Key Features:
- 5-Fold Cross Validation with stratified splitting
- Two-Phase Training Strategy (frozen backbone → fine-tuning)
- Automated Hyperparameter Tuning with Optuna (50 trials per model)
- Threshold Optimization for sensitivity-focused classification
- Comprehensive evaluation metrics and visualizations
- Production-ready checkpointing and model management

**Training Date:** Initiated on notebook run
**Target Task:** Binary Classification (Positive/Negative)

## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, roc_auc_score, classification_report
)
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import efficientnet, resnet50, convnext

import optuna
from optuna.trial import Trial
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Configure GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Configuration Parameters
CONFIG = {
    'dataset_dir': r'E:\DFU\Model\Dataset',
    'img_dir': r'E:\DFU\Model\Dataset\IMG',
    'roi_dir': r'E:\DFU\Model\Dataset\ROI',
    'checkpoint_dir': './model_checkpoints',
    'results_dir': './results',
    'img_size': (224, 224),
    'batch_size_default': 32,
    'max_epochs': 100,
    'phase1_patience': 5,
    'phase2_patience': 15,
    'n_folds': 5,
    'test_split': 0.2,
    'optuna_trials': 50,
    'phase2_lr_decay': 0.95,
    'phase2_decay_steps': 10000,
}

# Create output directories
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)

# Initialize logging
log_file = os.path.join(CONFIG['results_dir'], f"training_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

def log_message(message: str, print_also: bool = True):
    """Log message to file and optionally to console"""
    with open(log_file, 'a') as f:
        f.write(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}\n")
    if print_also:
        print(message)

log_message("="*80)
log_message("DFU MODEL TRAINING PIPELINE INITIALIZED")
log_message(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Data Loading & Preprocessing

In [ ]:
def load_dataset_from_structure(img_dir: str, img_size: Tuple[int, int] = (224, 224)):
    """
    Load images from new dataset structure.
    Expected structure:
    - img_dir/
        - DM/       (Diabetic at-risk, label=1)
            - P###_L.png
            - P###_R.png
        - CT/       (Control normal, label=0)
            - P###_L.png
            - P###_R.png
    
    Returns:
        images: numpy array of shape (N, H, W, 3)
        labels: numpy array of binary labels (0: CT normal, 1: DM at-risk)
        image_names: list of original image filenames
        image_sources: list indicating 'DM' or 'CT' source
    """
    images = []
    labels = []
    image_names = []
    image_sources = []
    
    if not os.path.exists(img_dir):
        raise FileNotFoundError(f"Image directory not found: {img_dir}")
    
    # Load DM images (label=1)
    dm_path = os.path.join(img_dir, 'DM')
    if os.path.exists(dm_path):
        for img_name in sorted(os.listdir(dm_path)):
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                img_path = os.path.join(dm_path, img_name)
                try:
                    img = load_img(img_path, target_size=img_size)
                    img_array = img_to_array(img) / 255.0  # Normalize to [0,1]
                    images.append(img_array)
                    labels.append(1)  # DM = positive/at-risk
                    image_names.append(img_name)
                    image_sources.append('DM')
                except Exception as e:
                    log_message(f"Error loading {img_path}: {str(e)}")
    
    # Load CT images (label=0)
    ct_path = os.path.join(img_dir, 'CT')
    if os.path.exists(ct_path):
        for img_name in sorted(os.listdir(ct_path)):
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                img_path = os.path.join(ct_path, img_name)
                try:
                    img = load_img(img_path, target_size=img_size)
                    img_array = img_to_array(img) / 255.0  # Normalize to [0,1]
                    images.append(img_array)
                    labels.append(0)  # CT = negative/normal
                    image_names.append(img_name)
                    image_sources.append('CT')
                except Exception as e:
                    log_message(f"Error loading {img_path}: {str(e)}")
    
    if len(images) == 0:
        raise ValueError(f"No images found in {img_dir}. Check directory structure.")
    
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    
    log_message(f"Loaded {len(images)} images from dataset structure")
    log_message(f"  - Shape: {images.shape}")
    log_message(f"  - Label distribution: {np.bincount(labels)}")
    log_message(f"  - DM (at-risk, label=1): {np.sum(labels == 1)}")
    log_message(f"  - CT (normal, label=0): {np.sum(labels == 0)}")
    log_message(f"  - Class balance: {np.sum(labels == 1) / len(labels) * 100:.2f}% DM at-risk")
    
    return images, labels, image_names, image_sources

# Load data
log_message("\nLoading images from dataset structure...")
log_message(f"Image directory: {CONFIG['img_dir']}")
try:
    images, labels, image_names, image_sources = load_dataset_from_structure(
        CONFIG['img_dir'], 
        img_size=CONFIG['img_size']
    )
    log_message(f"✓ Data loading successful: {images.shape[0]} samples")
except Exception as e:
    log_message(f"✗ Error loading data: {str(e)}")
    raise

# Verify data integrity
assert images.shape[1:] == (224, 224, 3), "Image size mismatch"
assert np.min(images) >= 0 and np.max(images) <= 1, "Image normalization issue"
assert len(np.unique(labels)) == 2, "Expected binary classification"

log_message(f"\nData verification:")
log_message(f"  - Min/Max pixel values: {images.min():.4f} / {images.max():.4f}")
log_message(f"  - DM at-risk (1): {np.sum(labels == 1)} | CT normal (0): {np.sum(labels == 0)}")
log_message(f"  - Class balance: {np.sum(labels == 1) / len(labels) * 100:.2f}% DM at-risk")

## 3. Train/Val/Test Split Logic

In [ ]:
def create_fold_splits(images: np.ndarray, labels: np.ndarray, n_splits: int = 5, 
                       test_split: float = 0.2, random_state: int = SEED):
    """
    Create 5-fold cross-validation splits with stratification.
    Each fold: 80% train/val, 20% held-out test.
    """
    # First split: 80% for CV, 20% for final test set
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    
    fold_indices = []
    test_indices = None
    
    # Get first split as test set
    for fold_idx, (train_val_idx, test_idx) in enumerate(skf.split(images, labels)):
        if fold_idx == 0:
            test_indices = test_idx
            break
    
    # Create 5 folds from remaining data (excluding test set)
    train_val_images = images[train_val_idx]
    train_val_labels = labels[train_val_idx]
    train_val_original_idx = train_val_idx
    
    # Create 5 folds with stratification
    skf_folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf_folds.split(train_val_images, train_val_labels)):
        # Map back to original indices
        train_original = train_val_original_idx[train_idx]
        val_original = train_val_original_idx[val_idx]
        
        fold_indices.append({
            'fold': fold_idx,
            'train_idx': train_original,
            'val_idx': val_original
        })
    
    return fold_indices, test_indices

# Create folds
log_message("\nCreating 5-fold cross-validation splits...")
fold_indices, test_indices = create_fold_splits(
    images, labels,
    n_splits=CONFIG['n_folds'],
    test_split=CONFIG['test_split'],
    random_state=SEED
)

# Prepare test set
X_test = images[test_indices]
y_test = labels[test_indices]

log_message(f"✓ Created {len(fold_indices)} folds")
log_message(f"  - Test set size: {len(y_test)} (labels: {np.bincount(y_test)})")

for fold_info in fold_indices[:3]:  # Show first 3 folds
    fold_num = fold_info['fold']
    train_size = len(fold_info['train_idx'])
    val_size = len(fold_info['val_idx'])
    log_message(f"  - Fold {fold_num}: train={train_size}, val={val_size}")

# Store original images for use in training
X_full = images.copy()
y_full = labels.copy()

log_message("✓ Data split creation complete")

## 4. Custom Model Class with Two-Phase Training

In [ ]:
class ExponentialDecayScheduler(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Custom learning rate schedule for Phase 2: lr_t = lr_0 * 0.95^(step/10000)"""
    
    def __init__(self, initial_lr: float, decay_rate: float = 0.95, decay_steps: int = 10000):
        self.initial_lr = initial_lr
        self.decay_rate = decay_rate
        self.decay_steps = decay_steps
    
    def __call__(self, step):
        return self.initial_lr * tf.pow(self.decay_rate, tf.cast(step, tf.float32) / self.decay_steps)
    
    def get_config(self):
        return {
            "initial_lr": self.initial_lr,
            "decay_rate": self.decay_rate,
            "decay_steps": self.decay_steps,
        }


class DFUModelTrainer:
    """Two-phase training strategy for transfer learning models"""
    
    def __init__(self, model_name: str, base_model, dropout_rate: float = 0.5, 
                 l2_reg: float = 1e-5, dense_units: Tuple[int, int] = (256, 64)):
        """
        Initialize model with custom head.
        
        Args:
            model_name: Name of the model architecture
            base_model: Pre-trained base model (without top layers)
            dropout_rate: Dropout rate for regularization
            l2_reg: L2 regularization coefficient
            dense_units: Tuple of (dense_layer1_units, dense_layer2_units)
        """
        self.model_name = model_name
        self.base_model = base_model
        self.dropout_rate = dropout_rate
        self.l2_reg = l2_reg
        self.dense_units = dense_units
        self.model = None
        self.history = None
        self.phase1_history = None
        self.phase2_history = None
    
    def build_model(self):
        """Build model with custom head"""
        # Freeze base model
        self.base_model.trainable = False
        
        # Build custom head
        inputs = layers.Input(shape=(224, 224, 3))
        x = self.base_model(inputs, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(self.dense_units[0], activation='relu',
                        kernel_regularizer=keras.regularizers.l2(self.l2_reg))(x)
        x = layers.Dropout(self.dropout_rate)(x)
        x = layers.Dense(self.dense_units[1], activation='relu',
                        kernel_regularizer=keras.regularizers.l2(self.l2_reg))(x)
        x = layers.Dropout(self.dropout_rate)(x)
        outputs = layers.Dense(1, activation='sigmoid')(x)
        
        self.model = models.Model(inputs=inputs, outputs=outputs)
        return self.model
    
    def train_phase1(self, X_train, y_train, X_val, y_val, batch_size: int = 32,
                     optimizer: str = 'adam', learning_rate: float = 1e-3,
                     max_epochs: int = 100, patience: int = 5, verbose: int = 1):
        """
        Phase 1: Train custom head with frozen backbone
        
        Early stopping with patience=5 epochs
        """
        log_message(f"\n{'='*80}")
        log_message(f"PHASE 1: Training Custom Head (Frozen Backbone) - {self.model_name}")
        log_message(f"{'='*80}")
        
        # Compile with frozen backbone
        if optimizer.lower() == 'adam':
            opt = Adam(learning_rate=learning_rate)
        elif optimizer.lower() == 'rmsprop':
            opt = RMSprop(learning_rate=learning_rate)
        else:
            opt = SGD(learning_rate=learning_rate, momentum=0.9)
        
        self.model.compile(
            optimizer=opt,
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        # Class weights for balanced training
        class_weights = {
            0: 1.0 / (np.sum(y_train == 0) / len(y_train)),
            1: 1.0 / (np.sum(y_train == 1) / len(y_train))
        }
        
        # Callbacks
        checkpoint_path = os.path.join(
            CONFIG['checkpoint_dir'],
            f"{self.model_name}_phase1_fold_best.h5"
        )
        
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True,
                verbose=1
            ),
            ModelCheckpoint(
                checkpoint_path,
                monitor='val_loss',
                save_best_only=True,
                verbose=0
            )
        ]
        
        # Train Phase 1
        self.phase1_history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            batch_size=batch_size,
            epochs=max_epochs,
            class_weight=class_weights,
            callbacks=callbacks,
            verbose=verbose
        )
        
        log_message(f"Phase 1 training complete. Best val_loss: {min(self.phase1_history.history['val_loss']):.6f}")
    
    def train_phase2(self, X_train, y_train, X_val, y_val, batch_size: int = 32,
                     optimizer: str = 'adam', learning_rate: float = 1e-4,
                     max_epochs: int = 100, patience: int = 15, verbose: int = 1):
        """
        Phase 2: Fine-tune with unfrozen top layers using exponential decay
        
        Learning rate schedule: lr_t = lr_0 × 0.95^(t/10000)
        Early stopping with patience=15 epochs (continuing from Phase 1)
        """
        log_message(f"\n{'='*80}")
        log_message(f"PHASE 2: Fine-tuning with Top Layers Unfrozen - {self.model_name}")
        log_message(f"{'='*80}")
        
        # Unfreeze top layers of backbone
        num_layers = len(self.base_model.layers)
        unfreeze_from = int(num_layers * 0.7)  # Unfreeze top 30% of layers
        
        for layer in self.base_model.layers[unfreeze_from:]:
            layer.trainable = True
        
        log_message(f"Unfroze {num_layers - unfreeze_from} layers (from layer {unfreeze_from})")
        
        # Compile with exponential decay
        lr_schedule = ExponentialDecayScheduler(
            initial_lr=learning_rate,
            decay_rate=CONFIG['phase2_lr_decay'],
            decay_steps=CONFIG['phase2_decay_steps']
        )
        
        if optimizer.lower() == 'adam':
            opt = Adam(learning_rate=lr_schedule)
        elif optimizer.lower() == 'rmsprop':
            opt = RMSprop(learning_rate=lr_schedule)
        else:
            opt = SGD(learning_rate=lr_schedule, momentum=0.9)
        
        self.model.compile(
            optimizer=opt,
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        # Class weights
        class_weights = {
            0: 1.0 / (np.sum(y_train == 0) / len(y_train)),
            1: 1.0 / (np.sum(y_train == 1) / len(y_train))
        }
        
        # Callbacks
        checkpoint_path = os.path.join(
            CONFIG['checkpoint_dir'],
            f"{self.model_name}_phase2_fold_best.h5"
        )
        
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True,
                verbose=1
            ),
            ModelCheckpoint(
                checkpoint_path,
                monitor='val_loss',
                save_best_only=True,
                verbose=0
            )
        ]
        
        # Train Phase 2
        self.phase2_history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            batch_size=batch_size,
            epochs=max_epochs,
            class_weight=class_weights,
            callbacks=callbacks,
            verbose=verbose
        )
        
        log_message(f"Phase 2 training complete. Best val_loss: {min(self.phase2_history.history['val_loss']):.6f}")
        log_message(f"{'='*80}\n")
    
    def get_predictions(self, X):
        """Get model predictions"""
        return self.model.predict(X, verbose=0).flatten()
    
    def save_model(self, path: str):
        """Save model to disk"""
        self.model.save(path)
        log_message(f"Model saved to {path}")

## 5. Hyperparameter Tuning with Optuna

In [ ]:
class OptunaHyperparameterTuner:
    """Optuna-based hyperparameter optimization for DFU models"""
    
    def __init__(self, model_name: str, base_model_fn, n_trials: int = 50):
        """
        Initialize tuner.
        
        Args:
            model_name: Name of the model
            base_model_fn: Function to create base model
            n_trials: Number of trials to run
        """
        self.model_name = model_name
        self.base_model_fn = base_model_fn
        self.n_trials = n_trials
        self.best_params = None
        self.best_value = 0
        self.study = None
    
    def objective(self, trial: Trial, X_train: np.ndarray, y_train: np.ndarray,
                  X_val: np.ndarray, y_val: np.ndarray) -> float:
        """Optuna objective function"""
        
        # Define search space
        dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.7)
        l2_reg = trial.suggest_categorical('l2_reg', [1e-6, 1e-5, 1e-4])
        dense_units_1 = trial.suggest_categorical('dense_units_1', [128, 256, 512])
        dense_units_2 = trial.suggest_categorical('dense_units_2', [32, 64, 128])
        batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
        optimizer = trial.suggest_categorical('optimizer', ['adam', 'rmsprop', 'sgd'])
        phase1_lr = trial.suggest_float('phase1_lr', 1e-4, 1e-2, log=True)
        phase2_lr = trial.suggest_float('phase2_lr', 1e-5, 1e-3, log=True)
        
        try:
            # Build model
            base_model = self.base_model_fn()
            trainer = DFUModelTrainer(
                model_name=self.model_name,
                base_model=base_model,
                dropout_rate=dropout_rate,
                l2_reg=l2_reg,
                dense_units=(dense_units_1, dense_units_2)
            )
            trainer.build_model()
            
            # Phase 1 training
            trainer.train_phase1(
                X_train, y_train, X_val, y_val,
                batch_size=batch_size,
                optimizer=optimizer,
                learning_rate=phase1_lr,
                max_epochs=50,
                patience=5,
                verbose=0
            )
            
            # Phase 2 training
            trainer.train_phase2(
                X_train, y_train, X_val, y_val,
                batch_size=batch_size,
                optimizer=optimizer,
                learning_rate=phase2_lr,
                max_epochs=50,
                patience=15,
                verbose=0
            )
            
            # Evaluate
            val_preds = trainer.get_predictions(X_val)
            val_auc = roc_auc_score(y_val, val_preds)
            
            return val_auc
        
        except Exception as e:
            log_message(f"Trial failed with error: {str(e)}")
            return 0.0
    
    def optimize(self, X_train: np.ndarray, y_train: np.ndarray,
                 X_val: np.ndarray, y_val: np.ndarray):
        """Run Optuna optimization"""
        
        log_message(f"\n{'='*80}")
        log_message(f"OPTUNA HYPERPARAMETER TUNING: {self.model_name}")
        log_message(f"Number of trials: {self.n_trials}")
        log_message(f"{'='*80}")
        
        sampler = TPESampler(seed=SEED)
        self.study = optuna.create_study(
            direction='maximize',
            sampler=sampler
        )
        
        # Optimize
        self.study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=self.n_trials,
            show_progress_bar=True
        )
        
        self.best_params = self.study.best_params
        self.best_value = self.study.best_value
        
        log_message(f"\n✓ Optimization complete")
        log_message(f"Best validation AUC: {self.best_value:.6f}")
        log_message(f"Best hyperparameters:")
        for key, value in self.best_params.items():
            log_message(f"  - {key}: {value}")
        
        return self.best_params


def create_base_models():
    """Create base models with ImageNet weights"""
    def efficientnet_b0():
        return efficientnet.EfficientNetB0(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
    
    def resnet50():
        return resnet50.ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
    
    def convnext_tiny():
        return convnext.ConvNeXtTiny(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
    
    return {
        'EfficientNetB0': efficientnet_b0,
        'ResNet50': resnet50,
        'ConvNeXt-Tiny': convnext_tiny
    }

base_model_creators = create_base_models()

log_message("✓ Optuna tuner and base model creators initialized")

## 6. Train EfficientNetB0 Model

In [ ]:
# Training EfficientNetB0
model_name_efficientnet = 'EfficientNetB0'
efficientnet_results = {
    'fold_models': [],
    'fold_histories': [],
    'fold_val_predictions': [],
    'fold_best_thresholds': [],
    'fold_metrics': []
}

log_message(f"\n{'#'*80}")
log_message(f"# TRAINING {model_name_efficientnet}")
log_message(f"{'#'*80}\n")

for fold_num, fold_info in enumerate(fold_indices):
    log_message(f"\n{'='*80}")
    log_message(f"FOLD {fold_num + 1}/{len(fold_indices)}: {model_name_efficientnet}")
    log_message(f"{'='*80}")
    
    # Get fold data
    train_idx = fold_info['train_idx']
    val_idx = fold_info['val_idx']
    
    X_train_fold = X_full[train_idx]
    y_train_fold = y_full[train_idx]
    X_val_fold = X_full[val_idx]
    y_val_fold = y_full[val_idx]
    
    log_message(f"Train set: {len(X_train_fold)} samples (DFU: {np.sum(y_train_fold)}, Normal: {np.sum(y_train_fold == 0)})")
    log_message(f"Val set: {len(X_val_fold)} samples (DFU: {np.sum(y_val_fold)}, Normal: {np.sum(y_val_fold == 0)})")
    
    # Step 1: Hyperparameter tuning
    tuner = OptunaHyperparameterTuner(
        model_name=f"{model_name_efficientnet}_Fold{fold_num + 1}",
        base_model_fn=base_model_creators['EfficientNetB0'],
        n_trials=CONFIG['optuna_trials']
    )
    
    best_params = tuner.optimize(X_train_fold, y_train_fold, X_val_fold, y_val_fold)
    
    # Step 2: Train final model with best hyperparameters
    log_message(f"\nTraining final model with best hyperparameters...")
    
    base_model = base_model_creators['EfficientNetB0']()
    trainer = DFUModelTrainer(
        model_name=f"{model_name_efficientnet}_Fold{fold_num + 1}",
        base_model=base_model,
        dropout_rate=best_params['dropout_rate'],
        l2_reg=best_params['l2_reg'],
        dense_units=(best_params['dense_units_1'], best_params['dense_units_2'])
    )
    
    trainer.build_model()
    
    # Phase 1
    trainer.train_phase1(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase1_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase1_patience'],
        verbose=1
    )
    
    # Phase 2
    trainer.train_phase2(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase2_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase2_patience'],
        verbose=1
    )
    
    # Save fold results
    val_predictions = trainer.get_predictions(X_val_fold)
    efficientnet_results['fold_models'].append(trainer.model)
    efficientnet_results['fold_histories'].append({
        'phase1': trainer.phase1_history.history,
        'phase2': trainer.phase2_history.history
    })
    efficientnet_results['fold_val_predictions'].append(val_predictions)
    
    # Save model checkpoint
    checkpoint_path = os.path.join(
        CONFIG['checkpoint_dir'],
        f"{model_name_efficientnet}_Fold{fold_num + 1}_final.h5"
    )
    trainer.save_model(checkpoint_path)
    
    log_message(f"✓ Fold {fold_num + 1} training complete")

log_message(f"\n{'#'*80}")
log_message(f"# {model_name_efficientnet} TRAINING COMPLETE")
log_message(f"{'#'*80}\n")

## 7. Train ResNet50 Model

In [ ]:
# Training ResNet50
model_name_resnet = 'ResNet50'
resnet_results = {
    'fold_models': [],
    'fold_histories': [],
    'fold_val_predictions': [],
    'fold_best_thresholds': [],
    'fold_metrics': []
}

log_message(f"\n{'#'*80}")
log_message(f"# TRAINING {model_name_resnet}")
log_message(f"{'#'*80}\n")

for fold_num, fold_info in enumerate(fold_indices):
    log_message(f"\n{'='*80}")
    log_message(f"FOLD {fold_num + 1}/{len(fold_indices)}: {model_name_resnet}")
    log_message(f"{'='*80}")
    
    # Get fold data
    train_idx = fold_info['train_idx']
    val_idx = fold_info['val_idx']
    
    X_train_fold = X_full[train_idx]
    y_train_fold = y_full[train_idx]
    X_val_fold = X_full[val_idx]
    y_val_fold = y_full[val_idx]
    
    log_message(f"Train set: {len(X_train_fold)} samples (DFU: {np.sum(y_train_fold)}, Normal: {np.sum(y_train_fold == 0)})")
    log_message(f"Val set: {len(X_val_fold)} samples (DFU: {np.sum(y_val_fold)}, Normal: {np.sum(y_val_fold == 0)})")
    
    # Step 1: Hyperparameter tuning
    tuner = OptunaHyperparameterTuner(
        model_name=f"{model_name_resnet}_Fold{fold_num + 1}",
        base_model_fn=base_model_creators['ResNet50'],
        n_trials=CONFIG['optuna_trials']
    )
    
    best_params = tuner.optimize(X_train_fold, y_train_fold, X_val_fold, y_val_fold)
    
    # Step 2: Train final model with best hyperparameters
    log_message(f"\nTraining final model with best hyperparameters...")
    
    base_model = base_model_creators['ResNet50']()
    trainer = DFUModelTrainer(
        model_name=f"{model_name_resnet}_Fold{fold_num + 1}",
        base_model=base_model,
        dropout_rate=best_params['dropout_rate'],
        l2_reg=best_params['l2_reg'],
        dense_units=(best_params['dense_units_1'], best_params['dense_units_2'])
    )
    
    trainer.build_model()
    
    # Phase 1
    trainer.train_phase1(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase1_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase1_patience'],
        verbose=1
    )
    
    # Phase 2
    trainer.train_phase2(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase2_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase2_patience'],
        verbose=1
    )
    
    # Save fold results
    val_predictions = trainer.get_predictions(X_val_fold)
    resnet_results['fold_models'].append(trainer.model)
    resnet_results['fold_histories'].append({
        'phase1': trainer.phase1_history.history,
        'phase2': trainer.phase2_history.history
    })
    resnet_results['fold_val_predictions'].append(val_predictions)
    
    # Save model checkpoint
    checkpoint_path = os.path.join(
        CONFIG['checkpoint_dir'],
        f"{model_name_resnet}_Fold{fold_num + 1}_final.h5"
    )
    trainer.save_model(checkpoint_path)
    
    log_message(f"✓ Fold {fold_num + 1} training complete")

log_message(f"\n{'#'*80}")
log_message(f"# {model_name_resnet} TRAINING COMPLETE")
log_message(f"{'#'*80}\n")

## 8. Train ConvNeXt-Tiny Model

In [ ]:
# Training ConvNeXt-Tiny
model_name_convnext = 'ConvNeXt-Tiny'
convnext_results = {
    'fold_models': [],
    'fold_histories': [],
    'fold_val_predictions': [],
    'fold_best_thresholds': [],
    'fold_metrics': []
}

log_message(f"\n{'#'*80}")
log_message(f"# TRAINING {model_name_convnext}")
log_message(f"{'#'*80}\n")

for fold_num, fold_info in enumerate(fold_indices):
    log_message(f"\n{'='*80}")
    log_message(f"FOLD {fold_num + 1}/{len(fold_indices)}: {model_name_convnext}")
    log_message(f"{'='*80}")
    
    # Get fold data
    train_idx = fold_info['train_idx']
    val_idx = fold_info['val_idx']
    
    X_train_fold = X_full[train_idx]
    y_train_fold = y_full[train_idx]
    X_val_fold = X_full[val_idx]
    y_val_fold = y_full[val_idx]
    
    log_message(f"Train set: {len(X_train_fold)} samples (DFU: {np.sum(y_train_fold)}, Normal: {np.sum(y_train_fold == 0)})")
    log_message(f"Val set: {len(X_val_fold)} samples (DFU: {np.sum(y_val_fold)}, Normal: {np.sum(y_val_fold == 0)})")
    
    # Step 1: Hyperparameter tuning
    tuner = OptunaHyperparameterTuner(
        model_name=f"{model_name_convnext}_Fold{fold_num + 1}",
        base_model_fn=base_model_creators['ConvNeXt-Tiny'],
        n_trials=CONFIG['optuna_trials']
    )
    
    best_params = tuner.optimize(X_train_fold, y_train_fold, X_val_fold, y_val_fold)
    
    # Step 2: Train final model with best hyperparameters
    log_message(f"\nTraining final model with best hyperparameters...")
    
    base_model = base_model_creators['ConvNeXt-Tiny']()
    trainer = DFUModelTrainer(
        model_name=f"{model_name_convnext}_Fold{fold_num + 1}",
        base_model=base_model,
        dropout_rate=best_params['dropout_rate'],
        l2_reg=best_params['l2_reg'],
        dense_units=(best_params['dense_units_1'], best_params['dense_units_2'])
    )
    
    trainer.build_model()
    
    # Phase 1
    trainer.train_phase1(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase1_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase1_patience'],
        verbose=1
    )
    
    # Phase 2
    trainer.train_phase2(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase2_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase2_patience'],
        verbose=1
    )
    
    # Save fold results
    val_predictions = trainer.get_predictions(X_val_fold)
    convnext_results['fold_models'].append(trainer.model)
    convnext_results['fold_histories'].append({
        'phase1': trainer.phase1_history.history,
        'phase2': trainer.phase2_history.history
    })
    convnext_results['fold_val_predictions'].append(val_predictions)
    
    # Save model checkpoint
    checkpoint_path = os.path.join(
        CONFIG['checkpoint_dir'],
        f"{model_name_convnext}_Fold{fold_num + 1}_final.h5"
    )
    trainer.save_model(checkpoint_path)
    
    log_message(f"✓ Fold {fold_num + 1} training complete")

log_message(f"\n{'#'*80}")
log_message(f"# {model_name_convnext} TRAINING COMPLETE")
log_message(f"{'#'*80}\n")

## 9. Threshold Optimization

In [ ]:
def optimize_threshold_per_fold(y_true, y_pred, target_sensitivity=0.85):
    """
    Optimize classification threshold to achieve target sensitivity.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted probabilities
        target_sensitivity: Minimum sensitivity to achieve (default 0.85)
    
    Returns:
        optimal_threshold: Best threshold achieving target sensitivity
        fpr, tpr, thresholds: ROC curve components
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    
    # Find threshold with sensitivity >= target_sensitivity
    valid_thresholds = []
    for i, threshold in enumerate(thresholds):
        sensitivity = tpr[i]
        if sensitivity >= target_sensitivity:
            specificity = 1 - fpr[i]
            valid_thresholds.append({
                'threshold': threshold,
                'sensitivity': sensitivity,
                'specificity': specificity,
                'distance': abs(sensitivity - 1) + abs(specificity - 1)  # Prioritize both
            })
    
    if not valid_thresholds:
        # If no threshold meets target sensitivity, use the one with highest sensitivity
        best_idx = np.argmax(tpr)
        optimal_threshold = thresholds[best_idx]
    else:
        # Select threshold that maximizes specificity while meeting sensitivity
        optimal_threshold = max(valid_thresholds, key=lambda x: x['specificity'])['threshold']
    
    return optimal_threshold, fpr, tpr, thresholds

# Perform threshold optimization for each model
log_message(f"\n{'='*80}")
log_message("THRESHOLD OPTIMIZATION")
log_message(f"{'='*80}\n")

# Process EfficientNetB0
log_message(f"Optimizing thresholds for {model_name_efficientnet}...")
for fold_num, (fold_info, val_preds) in enumerate(zip(fold_indices, efficientnet_results['fold_val_predictions'])):
    val_idx = fold_info['val_idx']
    y_val = y_full[val_idx]
    
    threshold, fpr, tpr, thresholds = optimize_threshold_per_fold(y_val, val_preds)
    efficientnet_results['fold_best_thresholds'].append(threshold)
    
    # Calculate metrics at optimal threshold
    y_pred_binary = (val_preds >= threshold).astype(int)
    sensitivity = recall_score(y_val, y_pred_binary)
    specificity = 1 - fpr[np.argmin(np.abs(thresholds - threshold))]
    
    log_message(f"  Fold {fold_num + 1}: Threshold={threshold:.4f}, Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")

# Process ResNet50
log_message(f"Optimizing thresholds for {model_name_resnet}...")
for fold_num, (fold_info, val_preds) in enumerate(zip(fold_indices, resnet_results['fold_val_predictions'])):
    val_idx = fold_info['val_idx']
    y_val = y_full[val_idx]
    
    threshold, fpr, tpr, thresholds = optimize_threshold_per_fold(y_val, val_preds)
    resnet_results['fold_best_thresholds'].append(threshold)
    
    # Calculate metrics at optimal threshold
    y_pred_binary = (val_preds >= threshold).astype(int)
    sensitivity = recall_score(y_val, y_pred_binary)
    specificity = 1 - fpr[np.argmin(np.abs(thresholds - threshold))]
    
    log_message(f"  Fold {fold_num + 1}: Threshold={threshold:.4f}, Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")

# Process ConvNeXt-Tiny
log_message(f"Optimizing thresholds for {model_name_convnext}...")
for fold_num, (fold_info, val_preds) in enumerate(zip(fold_indices, convnext_results['fold_val_predictions'])):
    val_idx = fold_info['val_idx']
    y_val = y_full[val_idx]
    
    threshold, fpr, tpr, thresholds = optimize_threshold_per_fold(y_val, val_preds)
    convnext_results['fold_best_thresholds'].append(threshold)
    
    # Calculate metrics at optimal threshold
    y_pred_binary = (val_preds >= threshold).astype(int)
    sensitivity = recall_score(y_val, y_pred_binary)
    specificity = 1 - fpr[np.argmin(np.abs(thresholds - threshold))]
    
    log_message(f"  Fold {fold_num + 1}: Threshold={threshold:.4f}, Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")

log_message(f"\n✓ Threshold optimization complete")

## 10. Evaluation & Results Comparison

In [ ]:
def evaluate_model_on_test_set(models, test_predictions, y_test, model_name, thresholds):
    """
    Evaluate model on test set using optimized thresholds.
    Aggregate predictions across folds.
    """
    # Ensemble: Average predictions from all folds
    ensemble_predictions = np.mean(test_predictions, axis=0)
    ensemble_threshold = np.mean(thresholds)
    
    # Binary predictions
    y_pred_binary = (ensemble_predictions >= ensemble_threshold).astype(int)
    
    # Calculate metrics
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred_binary),
        'precision': precision_score(y_test, y_pred_binary, zero_division=0),
        'recall': recall_score(y_test, y_pred_binary, zero_division=0),
        'f1': f1_score(y_test, y_pred_binary, zero_division=0),
        'sensitivity': recall_score(y_test, y_pred_binary, zero_division=0),
        'specificity': 1 - (np.sum((y_pred_binary == 1) & (y_test == 0)) / max(np.sum(y_test == 0), 1)),
        'auc_roc': roc_auc_score(y_test, ensemble_predictions),
        'ensemble_threshold': ensemble_threshold
    }
    
    return metrics, y_pred_binary, ensemble_predictions

# Get test predictions for each model
def get_test_predictions_ensemble(models, X_test):
    """Get predictions from all fold models"""
    test_preds = []
    for model in models:
        preds = model.predict(X_test, verbose=0).flatten()
        test_preds.append(preds)
    return np.array(test_preds)

log_message(f"\n{'='*80}")
log_message("TEST SET EVALUATION")
log_message(f"{'='*80}\n")

# Evaluate all models
all_models_results = {}

# EfficientNetB0
log_message(f"Evaluating {model_name_efficientnet}...")
test_preds_efficientnet = get_test_predictions_ensemble(
    efficientnet_results['fold_models'], X_test
)
metrics_efficientnet, y_pred_efficientnet, preds_efficientnet = evaluate_model_on_test_set(
    efficientnet_results['fold_models'],
    test_preds_efficientnet,
    y_test,
    model_name_efficientnet,
    efficientnet_results['fold_best_thresholds']
)
all_models_results[model_name_efficientnet] = metrics_efficientnet
efficientnet_results['test_metrics'] = metrics_efficientnet
efficientnet_results['test_predictions'] = y_pred_efficientnet
efficientnet_results['test_probabilities'] = preds_efficientnet

# ResNet50
log_message(f"Evaluating {model_name_resnet}...")
test_preds_resnet = get_test_predictions_ensemble(
    resnet_results['fold_models'], X_test
)
metrics_resnet, y_pred_resnet, preds_resnet = evaluate_model_on_test_set(
    resnet_results['fold_models'],
    test_preds_resnet,
    y_test,
    model_name_resnet,
    resnet_results['fold_best_thresholds']
)
all_models_results[model_name_resnet] = metrics_resnet
resnet_results['test_metrics'] = metrics_resnet
resnet_results['test_predictions'] = y_pred_resnet
resnet_results['test_probabilities'] = preds_resnet

# ConvNeXt-Tiny
log_message(f"Evaluating {model_name_convnext}...")
test_preds_convnext = get_test_predictions_ensemble(
    convnext_results['fold_models'], X_test
)
metrics_convnext, y_pred_convnext, preds_convnext = evaluate_model_on_test_set(
    convnext_results['fold_models'],
    test_preds_convnext,
    y_test,
    model_name_convnext,
    convnext_results['fold_best_thresholds']
)
all_models_results[model_name_convnext] = metrics_convnext
convnext_results['test_metrics'] = metrics_convnext
convnext_results['test_predictions'] = y_pred_convnext
convnext_results['test_probabilities'] = preds_convnext

# Create results dataframe
results_df = pd.DataFrame([all_models_results[model] for model in all_models_results.keys()])
results_df = results_df.set_index('model')

log_message("\n" + "="*80)
log_message("COMPREHENSIVE RESULTS TABLE")
log_message("="*80)
log_message("\n" + results_df.to_string())

# Save results to CSV
results_csv_path = os.path.join(CONFIG['results_dir'], 'model_comparison_results.csv')
results_df.to_csv(results_csv_path)
log_message(f"\n✓ Results saved to {results_csv_path}")

# Per-fold metrics
log_message(f"\n{'='*80}")
log_message("PER-FOLD DETAILED METRICS")
log_message(f"{'='*80}\n")

for fold_num in range(len(fold_indices)):
    log_message(f"\nFold {fold_num + 1}:")
    
    # Get validation data for this fold
    fold_info = fold_indices[fold_num]
    val_idx = fold_info['val_idx']
    y_val = y_full[val_idx]
    
    # EfficientNetB0
    val_preds_eff = efficientnet_results['fold_val_predictions'][fold_num]
    threshold_eff = efficientnet_results['fold_best_thresholds'][fold_num]
    y_pred_eff = (val_preds_eff >= threshold_eff).astype(int)
    
    auc_eff = roc_auc_score(y_val, val_preds_eff)
    f1_eff = f1_score(y_val, y_pred_eff, zero_division=0)
    
    log_message(f"  {model_name_efficientnet}: F1={f1_eff:.4f}, AUC={auc_eff:.4f}")
    
    # ResNet50
    val_preds_res = resnet_results['fold_val_predictions'][fold_num]
    threshold_res = resnet_results['fold_best_thresholds'][fold_num]
    y_pred_res = (val_preds_res >= threshold_res).astype(int)
    
    auc_res = roc_auc_score(y_val, val_preds_res)
    f1_res = f1_score(y_val, y_pred_res, zero_division=0)
    
    log_message(f"  {model_name_resnet}: F1={f1_res:.4f}, AUC={auc_res:.4f}")
    
    # ConvNeXt-Tiny
    val_preds_con = convnext_results['fold_val_predictions'][fold_num]
    threshold_con = convnext_results['fold_best_thresholds'][fold_num]
    y_pred_con = (val_preds_con >= threshold_con).astype(int)
    
    auc_con = roc_auc_score(y_val, val_preds_con)
    f1_con = f1_score(y_val, y_pred_con, zero_division=0)
    
    log_message(f"  {model_name_convnext}: F1={f1_con:.4f}, AUC={auc_con:.4f}")

log_message(f"\n✓ Evaluation complete")

## 11. Best Model Selection & Production Summary

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training History - Loss Comparison Across Folds (First 3 Folds)', fontsize=16, fontweight='bold')

# Plot training history for first 3 folds
for fold_num in range(min(3, len(fold_indices))):
    # EfficientNetB0
    ax = axes[0, fold_num]
    history_eff = efficientnet_results['fold_histories'][fold_num]
    ax.plot(history_eff['phase1']['loss'], label='Phase 1 - Train', linewidth=2)
    ax.plot(history_eff['phase1']['val_loss'], label='Phase 1 - Val', linewidth=2)
    ax.plot(np.arange(len(history_eff['phase1']['loss']), 
                      len(history_eff['phase1']['loss']) + len(history_eff['phase2']['loss'])),
            history_eff['phase2']['loss'], label='Phase 2 - Train', linewidth=2)
    ax.plot(np.arange(len(history_eff['phase1']['val_loss']), 
                      len(history_eff['phase1']['val_loss']) + len(history_eff['phase2']['val_loss'])),
            history_eff['phase2']['val_loss'], label='Phase 2 - Val', linewidth=2)
    ax.set_title(f'{model_name_efficientnet} - Fold {fold_num + 1}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # ResNet50
    ax = axes[1, fold_num]
    history_res = resnet_results['fold_histories'][fold_num]
    ax.plot(history_res['phase1']['loss'], label='Phase 1 - Train', linewidth=2)
    ax.plot(history_res['phase1']['val_loss'], label='Phase 1 - Val', linewidth=2)
    ax.plot(np.arange(len(history_res['phase1']['loss']), 
                      len(history_res['phase1']['loss']) + len(history_res['phase2']['loss'])),
            history_res['phase2']['loss'], label='Phase 2 - Train', linewidth=2)
    ax.plot(np.arange(len(history_res['phase1']['val_loss']), 
                      len(history_res['phase1']['val_loss']) + len(history_res['phase2']['val_loss'])),
            history_res['phase2']['val_loss'], label='Phase 2 - Val', linewidth=2)
    ax.set_title(f'{model_name_resnet} - Fold {fold_num + 1}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'training_history_first3folds.png'), dpi=300, bbox_inches='tight')
plt.show()

log_message("✓ Training history visualization saved")

# ROC Curves for Test Set
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ROC Curves - Test Set Ensemble Predictions', fontsize=14, fontweight='bold')

for idx, (model_name, preds, color) in enumerate([
    (model_name_efficientnet, efficientnet_results['test_probabilities'], 'blue'),
    (model_name_resnet, resnet_results['test_probabilities'], 'green'),
    (model_name_convnext, convnext_results['test_probabilities'], 'red')
]):
    ensemble_preds = np.mean(preds, axis=0)
    fpr, tpr, _ = roc_curve(y_test, ensemble_preds)
    auc_score = all_models_results[model_name]['auc_roc']
    
    axes[idx].plot(fpr, tpr, color=color, lw=3, label=f'AUC = {auc_score:.4f}')
    axes[idx].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    axes[idx].set_xlabel('False Positive Rate', fontsize=11)
    axes[idx].set_ylabel('True Positive Rate', fontsize=11)
    axes[idx].set_title(model_name, fontweight='bold')
    axes[idx].legend(loc='lower right', fontsize=10)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'roc_curves_test_set.png'), dpi=300, bbox_inches='tight')
plt.show()

log_message("✓ ROC curves visualization saved")

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices - Test Set', fontsize=14, fontweight='bold')

for idx, (model_name, y_pred) in enumerate([
    (model_name_efficientnet, efficientnet_results['test_predictions']),
    (model_name_resnet, resnet_results['test_predictions']),
    (model_name_convnext, convnext_results['test_predictions'])
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(model_name, fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'confusion_matrices_test_set.png'), dpi=300, bbox_inches='tight')
plt.show()

log_message("✓ Confusion matrices visualization saved")

# Model Comparison Bar Chart
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Model Performance Comparison - Test Set', fontsize=14, fontweight='bold')

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'sensitivity', 'auc_roc']
model_names = list(all_models_results.keys())
colors = ['#3498db', '#2ecc71', '#e74c3c']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    values = [all_models_results[m][metric] for m in model_names]
    bars = ax.bar(model_names, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_ylabel(metric.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'model_comparison_metrics.png'), dpi=300, bbox_inches='tight')
plt.show()

log_message("✓ Model comparison visualization saved")